# Agentic Alpha Position Sizing Search V1

本卡把 `agentic_alpha_ensemble_search_v1` 的多分支信号从“分支收益合成”转换成更接近执行的“SOXL 目标仓位”口径，并比较行业常见仓位管理方案：cap、迟滞、单日调仓上限、最小持有、波动率缩放，以及一个浅层概率门控模型。

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import evaluate_position_sizing_policies
from src.factor import (
    build_canary_sprint_frame,
    build_canary_trend_component,
    build_macro_credit_frame,
    build_macro_rate_gate_component,
    build_session_effect_frame,
    build_signed_overnight_component,
    build_tsmom_stop_component,
)
from soxl_backtest.reference import replay_soxlbacktest_reference

EXPERIMENT = 'agentic_alpha_position_sizing_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
MACRO_PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl_macro_credit_v1').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
SOURCE_START = '2010-03-11'
START = '2019-01-02'
END = '2026-06-09'
REPORT_START = '2026-01-02'
LATEST_SIGNAL_END = '2026-06-23'
MAX_DRAWDOWN_FLOOR = -20.0
COST_PER_TURNOVER = 0.0003
RAW_REFERENCE_RETURN_GATE = 8775.123636650942

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

selected_model = json.loads((ROOT / 'research' / 'agentic_alpha_ensemble_search_v1' / 'selected_model.json').read_text(encoding='utf-8'))
SELECTED_WEIGHTS = {str(key): float(value) for key, value in selected_model['weights'].items()}

daily_manifest = json.loads((PROVIDER_URI / 'qlib_store_manifest.json').read_text(encoding='utf-8'))
macro_manifest = json.loads((MACRO_PROVIDER_URI / 'qlib_store_manifest.json').read_text(encoding='utf-8'))
workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: position_sizing_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  macro_provider_uri: {MACRO_PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  research:
    start: {START}
    end: {END}
  report_window:
    start: {REPORT_START}
    end: {END}
position_sizing:
  cost_per_turnover: {COST_PER_TURNOVER}
  selected_reference: research/agentic_alpha_ensemble_search_v1/selected_model.json
  policies:
    - cap
    - hysteresis
    - turnover_limit
    - min_hold
    - volatility_target
    - shallow_probability_gate
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'qlib_docs',
            'title': 'Qlib workflow recorder',
            'url': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
            'extracted_signal': 'Use qlib workflow recorder for experiment evidence.',
        },
        {
            'source_type': 'qlib_docs',
            'title': 'Qlib portfolio strategy',
            'url': 'https://qlib.readthedocs.io/en/latest/component/strategy.html',
            'extracted_signal': 'Separate signal/model output from portfolio target position logic.',
        },
        {
            'source_type': 'industry_reference',
            'title': 'AQR Understanding Risk Parity',
            'url': 'https://www.aqr.com/-/media/AQR/Documents/Insights/White-Papers/Understanding-Risk-Parity.pdf',
            'extracted_signal': 'Risk budgeting changes position size as volatility changes.',
        },
        {
            'source_type': 'research_reference',
            'title': 'Volatility Targeting Is Trendy',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4773781',
            'extracted_signal': 'Volatility-managed equity strategies can overlap mechanically with trend following.',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)
print(json.dumps({'daily_as_of': daily_manifest['as_of'], 'macro_as_of': macro_manifest['as_of'], 'weights': SELECTED_WEIGHTS}, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "daily_as_of": "2026-06-23",
  "macro_as_of": "2026-06-09",
  "weights": {
    "canary_trend60_vix26_autocorr_h3_s1.0": 0.5144,
    "macro_rate_risk20_trend50_risk0_s0.5": 0.3241,
    "signed_overnight_mom20_n0.002_soxl60_vix45_s0.5": 0.3161,
    "tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02": 0.0405,
    "tsmom_soxx_ret10_th0_lowhv_q70_s1.0_stop0.02": 0.1473,
    "tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03": 0.1577
  }
}


In [2]:

def build_components(*, start: str, end: str, source_end: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
    daily_qlib_frame = D.features(
        ['SOXL', 'SOXX'],
        ['$open', '$high', '$low', '$close', '$volume', '$vix'],
        start_time=SOURCE_START,
        end_time=source_end,
        freq='day',
    )
    session_features = build_session_effect_frame(daily_qlib_frame, start=start, end=end, min_history=20)
    canary_features = build_canary_sprint_frame(daily_qlib_frame, start=start, end=end)
    daily_components = [
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s1.0_stop0.02', start=start, end=end, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=1.0, stop_loss=0.02, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03', start=start, end=end, score_name='soxx_ret252', threshold=0.15, regime_filter='vix30', scale=0.75, stop_loss=0.03, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02', start=start, end=end, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=0.75, stop_loss=0.02, lowhv_quantile=0.70),
        build_canary_trend_component(canary_features, component='canary_trend60_vix26_autocorr_h3_s1.0', start=start, end=end, trend_threshold=0.06, vix_max=26.0, hold_days=3, require_positive_autocorr=True, scale=1.0),
        build_signed_overnight_component(session_features, component='signed_overnight_mom20_n0.002_soxl60_vix45_s0.5', start=start, end=end, neutral_band=0.002, scale=0.5, momentum_window=20, trend_filter='soxl60', vix_max=45.0),
    ]

    qlib.init(provider_uri=MACRO_PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
    macro_qlib_frame = D.features(
        ['SOXL'],
        ['$soxl_close', '$soxx_close', '$yc10y2y', '$yc10y3m', '$credit_baa', '$baa_yield', '$hy_spread', '$dxy', '$oil'],
        start_time=start,
        end_time=END,
        freq='day',
    )
    macro_features = build_macro_credit_frame(macro_qlib_frame, start=start, end=END, min_history=20)
    macro_component = build_macro_rate_gate_component(
        macro_features,
        component='macro_rate_risk20_trend50_risk0_s0.5',
        start=start,
        end=END,
        risk_window=20,
        trend_window=50,
        risk_threshold=0.0,
        scale=0.5,
    )
    component_frame = pd.concat([*daily_components, macro_component], ignore_index=True)
    return component_frame, daily_qlib_frame, session_features, macro_features

component_frame, daily_qlib_frame, session_features, macro_features = build_components(start=START, end=END, source_end=LATEST_SIGNAL_END)
component_exposure = component_frame.pivot_table(index='datetime', columns='component', values='exposure', aggfunc='sum').sort_index().fillna(0.0)
weighted_exposure = component_exposure.copy()
for column in weighted_exposure.columns:
    weighted_exposure[column] = weighted_exposure[column] * SELECTED_WEIGHTS.get(column, 0.0)
raw_target = weighted_exposure.sum(axis=1).rename('raw_target')

normalized_daily = daily_qlib_frame.reset_index()
normalized_daily['instrument'] = normalized_daily['instrument'].astype(str).str.upper()
normalized_daily['datetime'] = pd.to_datetime(normalized_daily['datetime']).dt.normalize()
close_wide = normalized_daily.pivot_table(index='datetime', columns='instrument', values='$close', aggfunc='last').sort_index()
vix = normalized_daily.loc[normalized_daily['instrument'] == 'SOXL'].set_index('datetime')['$vix'].sort_index()
feature_frame = pd.DataFrame(index=close_wide.index)
feature_frame['soxl_ret1'] = close_wide['SOXL'].pct_change()
feature_frame['soxx_ret1'] = close_wide['SOXX'].pct_change()
feature_frame['soxl_hv20'] = feature_frame['soxl_ret1'].rolling(20, min_periods=20).std() * np.sqrt(252)
feature_frame['vix'] = vix
for window in [5, 10, 20, 60, 126, 252]:
    feature_frame[f'soxl_trend{window}'] = close_wide['SOXL'] / close_wide['SOXL'].shift(window) - 1.0
    feature_frame[f'soxx_trend{window}'] = close_wide['SOXX'] / close_wide['SOXX'].shift(window) - 1.0
feature_frame['forward_return'] = close_wide['SOXL'].pct_change().shift(-1)

macro_model_features = macro_features.set_index('datetime')[[
    'macro_risk_score20',
    'soxx_trend50',
    'soxl_trend50',
    'soxl_hv20',
    'rolling_drawdown42',
]].add_prefix('macro_')
position_frame = pd.DataFrame({'datetime': raw_target.index, 'raw_target': raw_target.values}).set_index('datetime')
position_frame = position_frame.join(weighted_exposure.add_prefix('leg_'), how='left').join(feature_frame, how='left').join(macro_model_features, how='left')
position_frame = position_frame.dropna(subset=['raw_target', 'forward_return']).reset_index()
report_frame = position_frame.loc[(position_frame['datetime'] >= pd.Timestamp(REPORT_START)) & (position_frame['datetime'] <= pd.Timestamp(END))].copy()
print(json.dumps({'component_rows': int(len(component_frame)), 'report_rows': int(len(report_frame)), 'report_start': str(report_frame['datetime'].min().date()), 'report_end': str(report_frame['datetime'].max().date())}, indent=2, sort_keys=True))


[40400:MainThread](2026-06-24 17:43:30,840) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40400:MainThread](2026-06-24 17:43:30,843) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40400:MainThread](2026-06-24 17:43:30,844) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[40400:MainThread](2026-06-24 17:43:30,916) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40400:MainThread](2026-06-24 17:43:30,917) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40400:MainThread](2026-06-24 17:43:30,918) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


{
  "component_rows": 11193,
  "report_end": "2026-06-09",
  "report_rows": 109,
  "report_start": "2026-01-02"
}


In [3]:

policies = [
    {'candidate': 'raw_target_position_reference', 'min_position': -2.0, 'max_position': 2.0, 'rebalance_threshold': 0.0, 'max_daily_change': None, 'min_hold_days': 1},
]
for max_position in [0.75, 1.0, 1.1, 1.25]:
    for threshold in [0.0, 0.1, 0.2, 0.3]:
        for max_daily_change in [None, 0.25, 0.5, 1.0]:
            for min_hold_days in [1, 2, 3]:
                change_label = max_daily_change if max_daily_change is not None else 'none'
                policies.append(
                    {
                        'candidate': f'cap{max_position:g}_h{threshold:g}_chg{change_label}_hold{min_hold_days}',
                        'min_position': 0.0,
                        'max_position': max_position,
                        'rebalance_threshold': threshold,
                        'max_daily_change': max_daily_change,
                        'min_hold_days': min_hold_days,
                    }
                )
for target_vol in [0.45, 0.60, 0.75, 0.90, 1.20]:
    for max_position in [0.75, 1.0, 1.1]:
        for threshold in [0.1, 0.2]:
            policies.append(
                {
                    'candidate': f'vol{target_vol:g}_cap{max_position:g}_h{threshold:g}',
                    'min_position': 0.0,
                    'max_position': max_position,
                    'rebalance_threshold': threshold,
                    'max_daily_change': 0.5,
                    'min_hold_days': 1,
                    'target_vol': target_vol,
                    'realized_vol_col': 'soxl_hv20',
                    'max_vol_scale': 1.0,
                }
            )

candidate_metrics, daily_positions = evaluate_position_sizing_policies(
    report_frame,
    policies=policies,
    raw_target_col='raw_target',
    return_col='forward_return',
    realized_vol_col='soxl_hv20',
    cost_per_turnover=COST_PER_TURNOVER,
)
legacy_replay = replay_soxlbacktest_reference(start=REPORT_START, end=END)
legacy_metrics = legacy_replay.metrics
raw_reference = candidate_metrics.loc[candidate_metrics['candidate'] == 'raw_target_position_reference'].iloc[0].to_dict()
candidate_metrics['passes_drawdown_gate'] = candidate_metrics['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR
candidate_metrics['beats_soxlbacktest'] = candidate_metrics['return_pct'] > float(legacy_metrics['rulebase_return_pct'])
candidate_metrics['beats_raw_position_reference'] = candidate_metrics['return_pct'] > float(raw_reference['return_pct'])
candidate_metrics['configured_exposure_budget_ok'] = candidate_metrics['candidate'].str.match(r'^(cap0\.75_|cap1_|vol.*_cap(0\.75|1)_h)')
candidate_metrics['exposure_budget_ok'] = (
    candidate_metrics['configured_exposure_budget_ok']
    & (candidate_metrics['max_position'] <= 1.0 + 1e-9)
    & (candidate_metrics['days_above_100'] == 0)
    & (candidate_metrics['days_below_0'] == 0)
)
candidate_metrics['managed_score'] = (
    candidate_metrics['return_pct']
    - candidate_metrics['next_day_reversal_count'] * 6.0
    - candidate_metrics['total_turnover']
    - candidate_metrics['days_above_100'] * 100.0
    - candidate_metrics['days_below_0'] * 100.0
)
managed_pool = candidate_metrics.loc[
    (candidate_metrics['candidate'] != 'raw_target_position_reference')
    & candidate_metrics['passes_drawdown_gate']
    & candidate_metrics['beats_soxlbacktest']
    & candidate_metrics['exposure_budget_ok']
]
selected_candidate = managed_pool.sort_values(['managed_score', 'return_pct'], ascending=[False, False]).iloc[0].to_dict()
import ast
selected_policy_params = ast.literal_eval(str(selected_candidate.get('params', '{}')))
print(candidate_metrics.sort_values('return_pct', ascending=False).head(15).to_string(index=False))
print(json.dumps({'selected': selected_candidate['candidate'], 'selected_return_pct': selected_candidate['return_pct'], 'raw_reference_return_pct': raw_reference['return_pct'], 'legacy_return_pct': legacy_metrics['rulebase_return_pct']}, indent=2, sort_keys=True))


[local] 读取 129 条数据
                    candidate  return_pct benchmark_return_pct  max_drawdown_pct  avg_abs_position  max_position  min_position  days_above_100  days_below_0  total_turnover  trade_count  next_day_reversal_count  avg_holding_days  cost_per_turnover                                                                                                                params  passes_drawdown_gate  beats_soxlbacktest  beats_raw_position_reference  configured_exposure_budget_ok  exposure_budget_ok  managed_score
raw_target_position_reference  479.156634                 None        -18.639418          0.614210       1.13045      -0.15805              13            13        9.519275           35                       11          3.176471             0.0003 {'min_position': -2.0, 'max_position': 2.0, 'rebalance_threshold': 0.0, 'max_daily_change': None, 'min_hold_days': 1}                  True                True                         False                          False         

In [4]:

# Shallow model gate: train on 2019-2025, infer 2026 only. This is deliberately a comparison, not the selected scheme.
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train_frame = position_frame.loc[position_frame['datetime'] < pd.Timestamp('2026-01-01')].copy()
test_frame = report_frame.copy()
feature_cols = [column for column in position_frame.columns if column not in {'datetime', 'forward_return'}]
train_frame = train_frame.dropna(subset=feature_cols + ['forward_return'])
train_medians = train_frame[feature_cols].median(numeric_only=True)
X_train = train_frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(train_medians)
X_test = test_frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(train_medians)
y_train_cls = (train_frame['forward_return'] > 0.0).astype(int)
y_train_reg = train_frame['forward_return'].astype(float)

model_candidates: list[pd.DataFrame] = []
models = {
    'logit_l2': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.3, class_weight='balanced')),
    'hgb_cls': HistGradientBoostingClassifier(max_iter=60, learning_rate=0.04, max_leaf_nodes=7, l2_regularization=0.3, random_state=7),
    'rf_cls': RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=40, class_weight='balanced_subsample', random_state=7),
}
base_eval = test_frame[['datetime', 'raw_target', 'forward_return', 'soxl_hv20']].copy()
for model_name, model in models.items():
    model.fit(X_train, y_train_cls)
    proba = model.predict_proba(X_test)[:, 1]
    for threshold in [0.45, 0.50, 0.55, 0.60, 0.65]:
        candidate_frame = base_eval.copy()
        candidate_frame['raw_target'] = candidate_frame['raw_target'].clip(0.0, 1.0) * (proba >= threshold).astype(float)
        model_metrics, _ = evaluate_position_sizing_policies(
            candidate_frame,
            policies=[
                {'candidate': f'{model_name}_gate{threshold:g}', 'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.1, 'max_daily_change': 1.0, 'min_hold_days': 1},
                {'candidate': f'{model_name}_gate{threshold:g}_h2', 'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.2, 'max_daily_change': 1.0, 'min_hold_days': 2},
            ],
            raw_target_col='raw_target',
            return_col='forward_return',
            realized_vol_col='soxl_hv20',
            cost_per_turnover=COST_PER_TURNOVER,
        )
        model_candidates.append(model_metrics)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
ridge.fit(X_train, y_train_reg)
ridge_pred = ridge.predict(X_test)
for threshold in [-0.002, 0.0, 0.002, 0.005]:
    candidate_frame = base_eval.copy()
    candidate_frame['raw_target'] = candidate_frame['raw_target'].clip(0.0, 1.0) * (ridge_pred >= threshold).astype(float)
    model_metrics, _ = evaluate_position_sizing_policies(
        candidate_frame,
        policies=[
            {'candidate': f'ridge_ret_gate{threshold:g}', 'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.1, 'max_daily_change': 1.0, 'min_hold_days': 1},
            {'candidate': f'ridge_ret_gate{threshold:g}_h2', 'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.2, 'max_daily_change': 1.0, 'min_hold_days': 2},
        ],
        raw_target_col='raw_target',
        return_col='forward_return',
        realized_vol_col='soxl_hv20',
        cost_per_turnover=COST_PER_TURNOVER,
    )
    model_candidates.append(model_metrics)

model_gate_metrics = pd.concat(model_candidates, ignore_index=True)
model_gate_metrics['passes_drawdown_gate'] = model_gate_metrics['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR
model_gate_metrics['beats_soxlbacktest'] = model_gate_metrics['return_pct'] > float(legacy_metrics['rulebase_return_pct'])
model_gate_metrics['beats_raw_position_reference'] = model_gate_metrics['return_pct'] > float(raw_reference['return_pct'])
best_model_gate = model_gate_metrics.sort_values('return_pct', ascending=False).iloc[0].to_dict()
model_position_gate = {
    'decision': 'not_primary',
    'train_window': ['2019-03-15', '2025-12-31'],
    'inference_window': [REPORT_START, END],
    'best_candidate': best_model_gate['candidate'],
    'best_return_pct': float(best_model_gate['return_pct']),
    'best_max_drawdown_pct': float(best_model_gate['max_drawdown_pct']),
    'best_next_day_reversal_count': int(best_model_gate['next_day_reversal_count']),
    'rationale': 'Shallow probability gates reduced drawdown but did not improve return versus the raw or constrained position reference.',
}
all_candidate_metrics = pd.concat([candidate_metrics, model_gate_metrics], ignore_index=True, sort=False)
print(pd.DataFrame([model_position_gate]).to_string(index=False))


   decision             train_window         inference_window          best_candidate  best_return_pct  best_max_drawdown_pct  best_next_day_reversal_count                                                                                                               rationale
not_primary [2019-03-15, 2025-12-31] [2026-01-02, 2026-06-09] ridge_ret_gate-0.002_h2       145.848909              -8.561831                             0 Shallow probability gates reduced drawdown but did not improve return versus the raw or constrained position reference.


In [5]:

# Latest signal overlay: use latest daily qlib store; macro leg is explicitly forward-filled from the stale macro provider.
latest_component_frame, latest_daily_qlib_frame, _, _ = build_components(start=START, end=LATEST_SIGNAL_END, source_end=LATEST_SIGNAL_END)
latest_exposure = latest_component_frame.pivot_table(index='datetime', columns='component', values='exposure', aggfunc='sum').sort_index().fillna(0.0)
macro_component = 'macro_rate_risk20_trend50_risk0_s0.5'
if macro_component in latest_exposure:
    latest_exposure[macro_component] = latest_exposure[macro_component].replace(0.0, np.nan).ffill().fillna(0.0)
latest_weighted = latest_exposure.copy()
for column in latest_weighted.columns:
    latest_weighted[column] = latest_weighted[column] * SELECTED_WEIGHTS.get(column, 0.0)
latest_raw_target = latest_weighted.sum(axis=1).rename('raw_target')
latest_policy_metrics, latest_policy_positions = evaluate_position_sizing_policies(
    pd.DataFrame({'datetime': latest_raw_target.index, 'raw_target': latest_raw_target.values, 'forward_return': 0.0, 'soxl_hv20': 1.0}),
    policies=[
        {
            'candidate': selected_candidate['candidate'],
            'min_position': float(selected_policy_params.get('min_position', 0.0)),
            'max_position': float(selected_policy_params.get('max_position', 1.0)),
            'rebalance_threshold': float(selected_policy_params.get('rebalance_threshold', 0.0)),
            'max_daily_change': selected_policy_params.get('max_daily_change'),
            'min_hold_days': int(selected_policy_params.get('min_hold_days', 1)),
        }
    ],
    raw_target_col='raw_target',
    return_col='forward_return',
    realized_vol_col='soxl_hv20',
    cost_per_turnover=0.0,
)
latest_row = latest_policy_positions.sort_values('datetime').iloc[-1]
latest_signal_overlay = {
    'as_of': pd.Timestamp(latest_row['datetime']).date().isoformat(),
    'raw_target_pct': float(latest_raw_target.iloc[-1] * 100.0),
    'selected_policy_target_pct': float(latest_row['target_position'] * 100.0),
    'selected_policy': selected_candidate['candidate'],
    'macro_credit_status': 'forward_filled_after_macro_provider_safe_end',
    'macro_provider_as_of': macro_manifest['as_of'],
    'daily_provider_as_of': daily_manifest['as_of'],
    'not_used_for_validated_return': True,
}
print(json.dumps(latest_signal_overlay, indent=2, sort_keys=True))


[40400:MainThread](2026-06-24 17:43:31,841) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40400:MainThread](2026-06-24 17:43:31,842) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40400:MainThread](2026-06-24 17:43:31,843) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[40400:MainThread](2026-06-24 17:43:31,968) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40400:MainThread](2026-06-24 17:43:31,969) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40400:MainThread](2026-06-24 17:43:31,969) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


{
  "as_of": "2026-06-23",
  "daily_provider_as_of": "2026-06-23",
  "macro_credit_status": "forward_filled_after_macro_provider_safe_end",
  "macro_provider_as_of": "2026-06-09",
  "not_used_for_validated_return": true,
  "raw_target_pct": 28.0325,
  "selected_policy": "cap1_h0.2_chgnone_hold1",
  "selected_policy_target_pct": 43.8375
}


In [6]:

candidate_metrics_path = CARD / 'position_sizing_candidate_metrics.csv'
selected_model_path = CARD / 'position_sizing_selected_model.json'
position_sizing_summary_path = CARD / 'position_sizing_summary.json'
benchmark_comparison_path = CARD / 'backtest_benchmark_comparison.csv'
failure_modes_path = CARD / 'failure_modes.md'
daily_positions_tmp = CARD / '.position_sizing_daily_positions.csv.gz'
model_metrics_tmp = CARD / '.position_sizing_model_gate_metrics.csv.gz'

all_candidate_metrics.to_csv(candidate_metrics_path, index=False)
selected_model_payload = {
    'candidate': selected_candidate['candidate'],
    'return_pct': float(selected_candidate['return_pct']),
    'max_drawdown_pct': float(selected_candidate['max_drawdown_pct']),
    'max_position': float(selected_candidate['max_position']),
    'days_above_100': int(selected_candidate['days_above_100']),
    'days_below_0': int(selected_candidate['days_below_0']),
    'next_day_reversal_count': int(selected_candidate['next_day_reversal_count']),
    'total_turnover': float(selected_candidate['total_turnover']),
    'selection_basis': 'highest managed_score among candidates beating soxl_backtest, passing drawdown gate, and respecting <=100% long-only exposure budget',
    'policy': selected_policy_params,
}
selected_model_path.write_text(json.dumps(selected_model_payload, indent=2, sort_keys=True), encoding='utf-8')
benchmark_comparison = pd.DataFrame(
    [
        {
            'benchmark': 'selected_position_policy',
            'return_pct': float(selected_candidate['return_pct']),
            'max_drawdown_pct': float(selected_candidate['max_drawdown_pct']),
            'trade_count': int(selected_candidate['trade_count']),
            'next_day_reversal_count': int(selected_candidate['next_day_reversal_count']),
            'max_position': float(selected_candidate['max_position']),
        },
        {
            'benchmark': 'raw_target_position_reference',
            'return_pct': float(raw_reference['return_pct']),
            'max_drawdown_pct': float(raw_reference['max_drawdown_pct']),
            'trade_count': int(raw_reference['trade_count']),
            'next_day_reversal_count': int(raw_reference['next_day_reversal_count']),
            'max_position': float(raw_reference['max_position']),
        },
        {
            'benchmark': 'soxl_backtest_reference',
            'return_pct': float(legacy_metrics['rulebase_return_pct']),
            'max_drawdown_pct': -float(legacy_metrics['rulebase_max_drawdown_pct']),
            'trade_count': int(legacy_metrics['trade_count']),
            'next_day_reversal_count': None,
            'max_position': float(legacy_replay.daily['position'].max()),
        },
        {
            'benchmark': 'soxl_buyhold',
            'return_pct': float(legacy_metrics['soxl_buyhold_return_pct']),
            'max_drawdown_pct': None,
            'trade_count': 1,
            'next_day_reversal_count': None,
            'max_position': 1.0,
        },
    ]
)
benchmark_comparison.to_csv(benchmark_comparison_path, index=False)

position_sizing_summary = {
    'workflow': EXPERIMENT,
    'goal_achieved_vs_soxlbacktest': bool(selected_candidate['return_pct'] > legacy_metrics['rulebase_return_pct'] and selected_candidate['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR),
    'goal_achieved_vs_raw_position_reference': bool(selected_candidate['return_pct'] > raw_reference['return_pct'] and selected_candidate['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR),
    'candidate_rows': int(len(all_candidate_metrics)),
    'selected_candidate': selected_model_payload,
    'raw_position_reference': {key: (float(value) if isinstance(value, (int, float, np.floating)) else value) for key, value in raw_reference.items() if key != 'params'},
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'model_position_gate': model_position_gate,
    'latest_signal_overlay': latest_signal_overlay,
}
position_sizing_summary_path.write_text(json.dumps(position_sizing_summary, indent=2, sort_keys=True), encoding='utf-8')
failure_modes_path.write_text(
    '# Failure Modes\n\n'
    '- 本卡把 ensemble 转成单一 SOXL 目标仓位口径，不能直接复现旧卡的分支收益合成 8775% 指标。\n'
    '- selected policy 超过 soxl_backtest YTD，但未超过 raw target-position reference；因此不直接晋升生产。\n'
    '- cap/hysteresis 降低了 >100% 暴露和隔日反向交易，但牺牲一部分收益。\n'
    '- latest signal overlay 使用 macro-credit stale/forward-filled 分支，只能读当前仓位，不能作为最新收益验证。\n'
    '- 浅层模型门控减少回撤但收益显著下降，神经网络仓位器需要更多样本和 walk-forward 约束后再考虑。\n',
    encoding='utf-8',
)
daily_positions.to_csv(daily_positions_tmp, index=False, compression='gzip')
model_gate_metrics.to_csv(model_metrics_tmp, index=False, compression='gzip')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder.log_params(
        provider_uri=PROVIDER_URI.as_posix(),
        macro_provider_uri=MACRO_PROVIDER_URI.as_posix(),
        report_start=REPORT_START,
        report_end=END,
        latest_signal_end=LATEST_SIGNAL_END,
        cost_per_turnover=COST_PER_TURNOVER,
        policy_rows=int(len(all_candidate_metrics)),
    )
    recorder.log_metrics(
        selected_return_pct=float(selected_candidate['return_pct']),
        selected_max_drawdown_pct=float(selected_candidate['max_drawdown_pct']),
        raw_position_reference_return_pct=float(raw_reference['return_pct']),
        soxlbacktest_return_pct=float(legacy_metrics['rulebase_return_pct']),
        latest_selected_target_pct=float(latest_signal_overlay['selected_policy_target_pct']),
    )
    for path in [candidate_metrics_path, selected_model_path, position_sizing_summary_path, benchmark_comparison_path, CARD / 'external_sources.csv', failure_modes_path]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_position_sizing')
    recorder.log_artifact(daily_positions_tmp.as_posix(), artifact_path='agentic_alpha_position_sizing')
    recorder.log_artifact(model_metrics_tmp.as_posix(), artifact_path='agentic_alpha_position_sizing')
    recorder_id = recorder.id
    recorder_local_dir = Path(recorder.uri).resolve()

daily_positions_tmp.unlink(missing_ok=True)
model_metrics_tmp.unlink(missing_ok=True)
position_sizing_summary['recorder_id'] = recorder_id
position_sizing_summary['recorder_local_dir'] = recorder_local_dir.as_posix()
position_sizing_summary['recorder_artifacts'] = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
position_sizing_summary_path.write_text(json.dumps(position_sizing_summary, indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps({'recorder_id': recorder_id, 'recorder_local_dir': recorder_local_dir.as_posix()}, indent=2, sort_keys=True))


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[40400:MainThread](2026-06-24 17:43:32,142) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_position_sizing_search_v1.


[40400:MainThread](2026-06-24 17:43:32,144) INFO - qlib.workflow - [exp.py:258] - Experiment 262966036026647502 starts running ...


[40400:MainThread](2026-06-24 17:43:32,213) INFO - qlib.workflow - [recorder.py:345] - Recorder fefeeea750c1494ba819e78642ede6ca starts running under Experiment 262966036026647502 ...


[40400:MainThread](2026-06-24 17:43:32,302) INFO - qlib.timer - [log.py:127] - Time cost: 0.004s | waiting `async_log` Done


{
  "recorder_id": "fefeeea750c1494ba819e78642ede6ca",
  "recorder_local_dir": "/Users/rziiiii/dev/workspace/research/agentic_alpha_position_sizing_search_v1/qlib_recorder"
}


In [7]:

artifact_policy = {
    'policy': 'slim_card_recorder_first',
    'workflow_config': 'workflow.yaml',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'kept_sidecars': [
        'backtest_benchmark_comparison.csv',
        'external_sources.csv',
        'failure_modes.md',
        'position_sizing_candidate_metrics.csv',
        'position_sizing_selected_model.json',
        'position_sizing_summary.json',
    ],
    'recorder_only_artifacts': {
        'predictions': 'artifacts/pred.pkl',
        'labels': 'artifacts/label.pkl',
        'signals': 'artifacts/signals.pkl',
        'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
        'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
        'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
        'daily_positions': 'artifacts/agentic_alpha_position_sizing/.position_sizing_daily_positions.csv.gz',
        'model_gate_metrics': 'artifacts/agentic_alpha_position_sizing/.position_sizing_model_gate_metrics.csv.gz',
    },
}
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'position_sizing_search',
    'promotion_decision': 'iterate_not_promoted_until_raw_position_reference_exceeded',
    'as_of': END,
    'latest_signal_as_of': LATEST_SIGNAL_END,
    'market': 'all',
    'region': 'us',
    'provider_uri': PROVIDER_URI.as_posix(),
    'macro_provider_uri': MACRO_PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'data_range': [START, END],
    'report_window': [REPORT_START, END],
    'goal_gate': {
        'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
        'require_return_above_soxlbacktest': True,
        'require_return_above_raw_position_reference': True,
    },
    'selected_candidate': selected_model_payload,
    'raw_position_reference': {key: (float(value) if isinstance(value, (int, float, np.floating)) else value) for key, value in raw_reference.items() if key != 'params'},
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'previous_branch_return_reference': {
        'candidate': selected_model['candidate'],
        'return_pct': float(selected_model['return_pct']),
        'max_drawdown_pct': float(selected_model['max_drawdown_pct']),
        'note': 'branch-return composite metric; not directly comparable to one target-position replay',
    },
    'model_position_gate': model_position_gate,
    'latest_signal_overlay': latest_signal_overlay,
    'position_sizing_report': position_sizing_summary,
    'data_freshness': {
        'daily_provider_as_of': daily_manifest['as_of'],
        'macro_provider_as_of': macro_manifest['as_of'],
        'validated_return_end': END,
        'latest_signal_end': LATEST_SIGNAL_END,
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
        'strategy': 'https://qlib.readthedocs.io/en/latest/component/strategy.html',
    },
    'workflow_config': 'workflow.yaml',
    'artifact_policy': artifact_policy,
    'sidecars': {
        'backtest_benchmark_comparison': 'backtest_benchmark_comparison.csv',
        'candidate_metrics': 'position_sizing_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'position_sizing_summary': 'position_sizing_summary.json',
        'selected_model': 'position_sizing_selected_model.json',
    },
    'recorder_artifacts': position_sizing_summary['recorder_artifacts'],
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
conclusion = (
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，用 qlib.init + D.features 重建 agentic ensemble 分支暴露，并把 raw signal 转成 SOXL target-position 口径比较。\n'
    f'- key_metrics: selected={selected_model_payload["candidate"]}，YTD return={selected_model_payload["return_pct"]:.4f}%，MaxDD={selected_model_payload["max_drawdown_pct"]:.4f}%，max_position={selected_model_payload["max_position"]:.2f}，days_above_100={selected_model_payload["days_above_100"]}。\n'
    f'- benchmark: soxl_backtest YTD return={legacy_metrics["rulebase_return_pct"]:.4f}%，MaxDD=-{legacy_metrics["rulebase_max_drawdown_pct"]:.4f}%；raw target-position reference return={raw_reference["return_pct"]:.4f}%，MaxDD={raw_reference["max_drawdown_pct"]:.4f}%。\n'
    f'- regime/window differences: cap + hysteresis 消除了超过 100% 的仓位，next_day_reversal 从 raw 的 {int(raw_reference["next_day_reversal_count"])} 降到 {selected_model_payload["next_day_reversal_count"]}，但未超过 raw target-position reference。\n'
    f'- model_view: 浅层概率门控最佳 return={model_position_gate["best_return_pct"]:.4f}%，MaxDD={model_position_gate["best_max_drawdown_pct"]:.4f}%，因此神经网络/模型仓位器当前不应作为第一优先级；更适合先做受约束仓位层。\n'
    f'- latest_signal: latest as_of={latest_signal_overlay["as_of"]}，raw_target={latest_signal_overlay["raw_target_pct"]:.4f}%，selected_policy_target={latest_signal_overlay["selected_policy_target_pct"]:.4f}%；macro 分支在 {macro_manifest["as_of"]} 后 forward-filled，只读仓位不验证收益。\n'
    '- promotion_decision: iterate_not_promoted_until_raw_position_reference_exceeded；该方案超过 soxl_backtest 且 MaxDD 小于 20%，但未超过 raw target-position reference，不能直接晋升。\n'
    f'- risks: 旧卡 8775% 是分支收益合成口径，不等于可执行单目标仓位；macro-credit provider 未更新到 daily provider 最新日期；仓位约束牺牲收益换取更低杠杆和更少隔日反向交易。qlib recorder id={recorder_id}。\n'
    '- next_action: 若要继续追求收益超过 raw reference，需要在仓位预算内重新搜索分支权重，或做 walk-forward 的 constrained optimizer，而不是直接上深度网络。\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)


# 结论

- what_ran: 执行 agentic_alpha_position_sizing_search_v1，用 qlib.init + D.features 重建 agentic ensemble 分支暴露，并把 raw signal 转成 SOXL target-position 口径比较。
- key_metrics: selected=cap1_h0.2_chgnone_hold1，YTD return=386.7833%，MaxDD=-19.3887%，max_position=1.00，days_above_100=0。
- benchmark: soxl_backtest YTD return=132.0780%，MaxDD=-13.5541%；raw target-position reference return=479.1566%，MaxDD=-18.6394%。
- regime/window differences: cap + hysteresis 消除了超过 100% 的仓位，next_day_reversal 从 raw 的 11 降到 1，但未超过 raw target-position reference。
- model_view: 浅层概率门控最佳 return=145.8489%，MaxDD=-8.5618%，因此神经网络/模型仓位器当前不应作为第一优先级；更适合先做受约束仓位层。
- latest_signal: latest as_of=2026-06-23，raw_target=28.0325%，selected_policy_target=43.8375%；macro 分支在 2026-06-09 后 forward-filled，只读仓位不验证收益。
- promotion_decision: iterate_not_promoted_until_raw_position_reference_exceeded；该方案超过 soxl_backtest 且 MaxDD 小于 20%，但未超过 raw target-position reference，不能直接晋升。
- risks: 旧卡 8775% 是分支收益合成口径，不等于可执行单目标仓位；macro-credit provider 未更新到 daily

## 中文结论

上一个代码单元同步写出 `summary.json` 和 `conclusion.md`，记录仓位管理候选、benchmark、latest signal overlay、是否晋升与下一步。